In [4]:

import pandas as pd
import numpy as np
import gc

from data.dataset import MalwareDatasetLoader
from data.data_processing import split_out_targets, force_dense, preprocess_existing, preprocess_fit

RELOAD_DATA = False
if not RELOAD_DATA:
  try:
    print(df_features_train.head())
  except Exception as e:
    print("No dataframe.  Loading data...")
    RELOAD_DATA=True
if RELOAD_DATA:
  df_loader = MalwareDatasetLoader()

  df_train, df_val, df_test = df_loader.make_data_splits()
  
  df_features_train, df_y_train = split_out_targets(df_train)
  df_features_val, df_y_val = split_out_targets(df_val)
  df_features_test, df_y_test = split_out_targets(df_test)
  del df_loader
  gc.collect()


   id.orig_p  id.resp_p  duration  orig_bytes  resp_bytes  missed_bytes  \
0    60836.0       23.0  3.145479         0.0         0.0           0.0   
1     8341.0    62336.0 -1.000000        -1.0        -1.0           0.0   
2    58186.0       23.0 -1.000000        -1.0        -1.0           0.0   
3    52808.0       23.0  3.122224         0.0         0.0           0.0   
4    50196.0       23.0  0.000001         0.0         0.0           0.0   

   orig_pkts  orig_ip_bytes  resp_pkts  resp_ip_bytes proto service conn_state  
0        6.0          360.0        0.0            0.0   tcp       -         S0  
1        0.0            0.0        0.0            0.0   tcp       -        OTH  
2        1.0           60.0        0.0            0.0   tcp       -         S0  
3        3.0          180.0        0.0            0.0   tcp       -         S0  
4        2.0          120.0        0.0            0.0   tcp       -         S0  


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def compute_metrics(classifier, df_features, df_y):
  print(f"Compute Metrics Start: {df_features.shape[0]}")
  predictions = classifier.predict(df_features)
  y_prob = classifier.predict_proba(df_features)[:, 1]
  print("Compute Metrics End")

  acc = accuracy_score(df_y, predictions)
  f1 = f1_score(df_y, predictions)
  auc = roc_auc_score(df_y, y_prob)
  cm = confusion_matrix(df_y, predictions)

  print(f"Accuracy: {acc:.4f}")
  print(f"F1:       {f1:.4f}")
  print(f"AUC:      {auc:.4f}")
  print("Confusion matrix:")
  print(cm)
  print()
  return classification_report(df_y, predictions)


In [13]:
RANDOM_STATE=2025

from data.data_processing import preprocess_fit

import sklearn

def train_bagging(df_features_train, df_y_train, max_depth=2, max_trees=50):
  tree_classifier = sklearn.tree.DecisionTreeClassifier(max_depth=max_depth)

  bagging_classifier = sklearn.ensemble.BaggingClassifier(
    estimator=tree_classifier,
    n_estimators=max_trees,
    max_samples=0.5,
    bootstrap=False,
    n_jobs=32,
    random_state=RANDOM_STATE)

  bagging_classifier.fit(df_features_train, df_y_train)

  return bagging_classifier

max_depth = 20
max_trees = 400
print("Training bagging model")
X_transformed, preprocessor = preprocess_fit(df_features_train, quantile_clipping=True)
bagging_classifier = train_bagging(X_transformed, df_y_train,
                                    max_depth=max_depth,
                                    max_trees=max_trees)
X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
print(metrics)

X_test_transformed = preprocess_existing(df_features_test, preprocessor)

metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
print(metrics_test)

Training bagging model


KeyboardInterrupt: 

Compute Metrics Start: 3751650
Compute Metrics End
Accuracy: 0.9929
F1:       0.9899
AUC:      0.9976
Confusion matrix:
[[2408565   26677]
 [    146 1316262]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2435242
           1       0.98      1.00      0.99   1316408

    accuracy                           0.99   3751650
   macro avg       0.99      0.99      0.99   3751650
weighted avg       0.99      0.99      0.99   3751650

Compute Metrics Start: 3751651
Compute Metrics End
Accuracy: 0.9929
F1:       0.9899
AUC:      0.9976
Confusion matrix:
[[2407926   26667]
 [    144 1316914]]

              precision    recall  f1-score   support

           0       1.00      0.99      0.99   2434593
           1       0.98      1.00      0.99   1317058

    accuracy                           0.99   3751651
   macro avg       0.99      0.99      0.99   3751651
weighted avg       0.99      0.99      0.99   3751651

